## Esame di Adv Sim Tech: Machine Learing nell'analisi dei Raggi Cosmici
*Autore: Roberto Di Loreto, matr. n° 302122*

Lo scopo di questo progetto è quello di verificare la validità e potenzialità di metodi di Machine Learing (ML) nella fisica dei raggi cosmici; in particolare, si vuole studiare come i metodi ML possano migliorare la ricostruzione dell'energia di eventi registrati dai telescopi del *Pierre Auger Observatory* (Pierre Auger Open Data).

I laboratori Pierre Auger utilizzano un metodo ricostruttivo dell'energia noto come Constant Intensity Cut (CIC). Questo metodo si basa su un'assunzione molto potente e delicata allo stesso tempo: il flusso di raggi cosmici è isotropo (stessa intensità in ogni direzione).

Da ciò deriva che per ogni angolo zenitale deve esserci un certo numero di eventi sopra una soglia fissata. Questa assunzione ha permesso di ricostruire una curva di attenuazione, ovvero di descrivere come segnali ad angoli maggiori subissero una diminuzione di intensità causata dal dover attraversare uno strato di atmosfera più lungo.
Il vantaggio principale del CIC sta nel non dipendere troppo dal modello teorico che descrive i raggi cosmici, infatti risulta molto efficace per eventi di tipo misto (protoni, nuclei pesanti)

Come anticipato, l'assunsione di flusso isotropo è delicata poiché non generale, infatti a scale più alte si osservano delle anomalie. L'altro problema principale sta nelle fluttuazioni naturali degli sciami cosmici, che portano ad imprecisioni in queste curve di attenuazione.

Proprio per questo motivo può essere istruttivo applicare metodi di ML che *affiancano l'attuale CIC* al fine di correggere evento per evento la ricostruzione di energia. Questa è l'idea su cui si basa il seguente progetto.

## Dataset Pierre Auger Open Data

I laboratori hanno pubblicato vari set di dati raccolti dal 2004 fino al 2021, reperibili al link https://opendata.auger.org/data.php. Sono presenti un file di pseudo-raw data ed uno sommario, contenente solo il 10% dei dati raccolti e revisionati secondo criteri di selezione che li rendono adatti alla maggior parte delle analisi statistiche. 

La prima operazione consiste nell'estrazione dati da questi files in formato ".csv", al fine di visionare la dimensione (n. di righe/colonne) e le caratteristiche principali.

In [1]:
import numpy as np
import pandas as pd
from progressbar import progress_bar
import csv
from pathlib import Path

# Usa la cartella corrente del notebook, che in Jupyter di solito è la cartella del file .ipynb

base_dir = Path.cwd()
dir_path = base_dir / "summary"
csv_files = sorted(dir_path.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"Nessun CSV trovato in: {dir_path}")

database = []
all_headers = []

for myfile in csv_files:
    with open(myfile, 'r') as f:
        reader = csv.reader(f)
        header = next(reader)  # Legge l'intestazione del file CSV
        all_headers.append(header)
        dataset = [[str(cell) for cell in row] for row in reader]  # Mantieni i valori come stringhe
        database.append(dataset)
    print("Done reading file:", myfile.name,'\n')
    print("Number of rows:", len(database[-1]),'\n')
    print(all_headers[-1],'\n','\n')       # Stampa la prima riga del dataset


Done reading file: dataSummaryInclined.csv 

Number of rows: 2355 

['id', 'sdid', 'gpstime', 'sd1500', 'multiEye', 'sd_gpsnanotime', 'sd_theta', 'sd_dtheta', 'sd_phi', 'sd_dphi', 'sd_energy', 'sd_denergy', 'sd_l', 'sd_b', 'sd_ra', 'sd_dec', 'sd_x', 'sd_dx', 'sd_y', 'sd_dy', 'sd_z', 'sd_easting', 'sd_northing', 'sd_altitude', 'sd_n19', 'sd_dn19', 'sd_n68', 'sd_dn68', 'sd_nbstat', 'sd_geondf', 'sd_geochi2', 'sd_exposure'] 
 

Done reading file: dataSummarySD1500.csv 

Number of rows: 24319 

['id', 'sdid', 'gpstime', 'sd1500', 'multiEye', 'sd_gpsnanotime', 'sd_theta', 'sd_dtheta', 'sd_phi', 'sd_dphi', 'sd_energy', 'sd_denergy', 'sd_l', 'sd_b', 'sd_ra', 'sd_dec', 'sd_x', 'sd_dx', 'sd_y', 'sd_dy', 'sd_z', 'sd_easting', 'sd_northing', 'sd_altitude', 'sd_R', 'sd_dR', 'sd_s1000', 'sd_ds1000', 'sd_s38', 'sd_gcorr', 'sd_wcorr', 'sd_beta', 'sd_gamma', 'sd_chi2', 'sd_ndf', 'sd_geochi2', 'sd_geondf', 'sd_nbstat', 'fd_id', 'fd_gpsnanotime', 'fd_hdSpectrumEye', 'fd_hdCalibEye', 'fd_hdXmaxEye', 'fd_

## Scelta del dataset

La cartella degli Open Data è suddivisa in tre files in base al tipo di misura effettuata per acquisirli. I primi due sfruttano dei *Surface Detectors* (SD) posti in una griglia a triangoli e distanti l'uno dall'altro rispettivamente 1500 metri e 750 metri; il terzo dataset presenta misure cosiddette "inclinate", ovvero ad angoli zenitali molto grandi (60°-80°), e riguarda principalmente la stima dell'energia del primario di uno sciame (non usa CIC).

Per lo scopo del progetto, il dataset SD750 potrebbe essere il compromesso utile, dato che l'array più denso genera un numero di dati sufficiente per far lavorare al meglio i modelli ML. Il dataset SD1500 potrebbe essere usato come test o per supporto, con le opportune accortezze.

## Analisi delle variabili chiave

Dopo aver individuato l'array di lavoro, il prossimo passo è cercare di capire quali variabili contiene il dataset, in modo da capire quali di queste potrebbero influenzare una ricostruzione shower-by-shower. Il metodo CIC sfrutta essenzialmente la ricostruzione del segnale ideale per l'array di riferimento (*s1000* per SD1500 e *s450* per SD750) e l'angolo zenitale associato ad ogni sciame (anzi il coseno quadro, legato all'intensità del segnale). Il file sommario ricostruito in Open Data presenta numerose righe con variabili aggiuntive a cui non è associato un valore, questo perché gli eventi selezionati sono principalmente osservati a terra da SD ma esistono detector a fluorescenza (FD) che servono a tracciare lo sciame in atmosfera.
\
Si può controllare se questi eventi formano un campione sufficiente da analizzare con ML.

***N.B.** il file manca della colonna 72, ovvero del valore SD38 che è il riferimento ottimale per stimare la curva di attenuazione. Questo fatto viene considerato nella ricerca delle righe "piene".*

In [2]:
# Scelta del dataset da utilizzare, ovvero SD750

myheader = all_headers[2]
mydata = database[2]

print("Header scelto:", myheader,'\n')

# Check sui dati SD-FD: seleziona le righe con esattamente s38 mancante
line = 0
other = 0

complete_rows = []
for row in mydata:
    empty_count = sum(1 for cell in row if cell == '')
    if empty_count == 1:
        complete_rows.append(row)
        missing_cols = [i for i, cell in enumerate(row) if cell == '']
        print("Riga completa trovata:    ID", row[0], "    Dato n. ", mydata.index(row))
    if empty_count in range(2,38):
        other += 1
        print("Riga anomala trovata:    ID", row[0] , "    Dato n.", mydata.index(row), "   mancanti:", empty_count)


print()
print("Totale righe con un solo valore mancante:", len(complete_rows))
print("Totale righe speciali:", len(complete_rows) + other)


Header scelto: ['id', 'sdid', 'gpstime', 'sd750', 'multiEye', 'sd1500', 'sd_gpsnanotime', 'sd_theta', 'sd_phi', 'sd_energy', 'sd_l', 'sd_b', 'sd_ra', 'sd_dec', 'sd_x', 'sd_dx', 'sd_y', 'sd_dy', 'sd_z', 'sd_easting', 'sd_northing', 'sd_altitude', 'sd_s450', 'sd_ds450', 'sd_s35', 'sd_beta', 'sd_gamma', 'sd_chi2', 'sd_ndf', 'sd_geochi2', 'sd_nbstat', 'fd_id', 'fd_gpsnanotime', 'fd_hdSpectrumEye', 'fd_hdCalibEye', 'fd_hdXmaxEye', 'fd_theta', 'fd_dtheta', 'fd_phi', 'fd_dphi', 'fd_l', 'fd_b', 'fd_ra', 'fd_dec', 'fd_totalEnergy', 'fd_dtotalEnergy', 'fd_calEnergy', 'fd_dcalEnergy', 'fd_xmax', 'fd_dxmax', 'fd_heightXmax', 'fd_distXmax', 'fd_dEdXmax', 'fd_ddEdXmax', 'fd_x', 'fd_dx', 'fd_y', 'fd_dy', 'fd_z', 'fd_easting', 'fd_northing', 'fd_altitude', 'fd_cherenkovFraction', 'fd_minViewAngle', 'fd_uspL', 'fd_duspL', 'fd_uspR', 'fd_duspR', 'fd_hottestStationId', 'fd_distSdpStation', 'fd_distAxisStation', 'sd_s38', 'sd_gpstime', 'sd_exposure'] 

Riga completa trovata:    ID 140035529400     Dato n.

**OSSERVAZIONI (1)**

Si può notare che il numero di eventi con righe complete è molto piccolo, per cui è consigliabile lavorare su tutti gli altri dati riducendo la matrice di partenza alle sole colonne "piene". Dato che le righe con valori mancanti sono tutte uguali, lavoro semplicemente sulla prima riga, selezionando le colonne da eliminare

In [3]:
# Riduzione del dataset ed eliminazione delle variabili con più di un valore mancante

# La variabile 'missing_cols' è calcolata nel punto precedente. La riscriviamo per il nostro caso.
missing_cols = [i for i, cell in enumerate(mydata[0]) if cell == '']
#print(missing_cols, '\n')

count = -1
for i in missing_cols:
    count += 1
    mydata = np.delete(mydata, i - count, axis=1)
    myheader = np.delete(myheader, i - count, axis=0)
    progress_bar(count, len(missing_cols) - 1)

print("     Fatto!", '\n')
print(myheader, '\n')
print("Variabili rimanenti", len(myheader))


████████████████████████████████████████████████████████████████████████████████ 100.00%     Fatto! 

['id' 'sdid' 'gpstime' 'sd750' 'multiEye' 'sd1500' 'sd_gpsnanotime'
 'sd_theta' 'sd_phi' 'sd_energy' 'sd_l' 'sd_b' 'sd_ra' 'sd_dec' 'sd_x'
 'sd_dx' 'sd_y' 'sd_dy' 'sd_z' 'sd_easting' 'sd_northing' 'sd_altitude'
 'sd_s450' 'sd_ds450' 'sd_s35' 'sd_beta' 'sd_gamma' 'sd_chi2' 'sd_ndf'
 'sd_geochi2' 'sd_nbstat' 'sd_gpstime' 'sd_exposure'] 

Variabili rimanenti 33


**OSSERVAZIONI (2)**

Sul sito del Pierre Auger Observatory sono indicate tutte le variabili misurate (dirrettamente o indirettamente) e contenute nei diversi file associati agli eventi. Nel sommario troviamo 33 di queste, che possiamo suddividere in variabili *info*, *flags*, *SD recostructed* (*sdrec*).

I 192 eventi trovati nella sezione precedente sono essenzialmente quelli con misure dei detector a fluorescenza (fdrec): sono noti come "golden-hybrid" events. Come suggerisce il nome, gli eventi di questo tipo sono di grande valore poiché hanno superato criteri di selezione di ben due classi di rivelatori. Possono essere gli eventi di base per una corretta calibrazione dei modelli ML.
\
Le variabili su cui i modelli ML potrebbero essere effettivamente usati sono proprio le *sdrec* Tra queste, notiamo la presenza degli errori associati alle misure, fondamentali nella ricostruzione in alcuni modelli.

Per comodità, separiamo il dataset rimanente in sotto-array di info, misure ed errori. Separiamo a parte l'esposizione (utile solo per flussi) ed il tempo gps.

In [4]:
# Separo gps ed exposure dal dataset principale
exposure = mydata[:, -1]
mydata = np.delete(mydata, -1, axis=1)
myheader = np.delete(myheader, -1, axis=0)

gps_time = mydata[:, -1]
mydata = np.delete(mydata, -1, axis=1)
myheader = np.delete(myheader, -1, axis=0)

# Check di eventuali buchi

for i in range(len(mydata)):
    if '' in mydata[i]:
        print("Riga con buchi trovata:")
        for j in range(len(mydata[i])):
            if mydata[i][j] == '':
                print("Colonna n.", j, "Valore mancante:", myheader[j])

Riga con buchi trovata:
Colonna n. 24 Valore mancante: sd_s35
Riga con buchi trovata:
Colonna n. 24 Valore mancante: sd_s35


Avendo determinato righe con valori mancanti in `sd_s35`, dato che questi non sono importanti per i processi successivi (ho già `sd_s450`), escludiamo anche questi dal dataset principale. In questo modo, il dataset finale sarà composto dalle sole 30 colonne utili per la ricostruzione dell'energia shower-by-shower.

In [5]:
# Separo s35
idx = np.where(myheader == 'sd_s35')[0][0]
sd35 = mydata[:, idx]
mydata = np.delete(mydata, idx, axis=1)
myheader = np.delete(myheader, idx, axis=0)

#print(myheader, '\n')

# Continuo con divisione in info, variabili, errori

mydata_info = mydata.copy()
mydata_var = mydata.copy()
mydata_err = mydata.copy()

# Le info sono le prime 7 entrate

infolist = ['','','','','','','']
for i in range(7):
    infolist[i] = myheader[i]

print("Info:", infolist, '\n')

errorlist = ['sd_dx', 'sd_dy', 'sd_ds450', 'sd_chi2', 'sd_ndf', 'sd_geochi2', 'sd_nbstat']
varlist = []

for i in range(len(myheader)):
    if myheader[i] not in infolist and myheader[i] not in errorlist:
        varlist.append(myheader[i])

mydata_info = np.delete(mydata_info, slice(7,len(mydata_info[0])), axis=1 )
mydata_err = np.delete(mydata_err, [np.where(myheader == cell)[0][0] for cell in myheader if cell not in errorlist], axis=1)
mydata_var = np.delete(mydata_var, [np.where(myheader == cell)[0][0] for cell in myheader if cell not in varlist], axis=1)

print("Variabili:", varlist, '\n')
print("Incertezze:", errorlist, '\n')

#Last check
for elem in myheader:
    if elem not in infolist and elem not in errorlist and elem not in varlist:
        print("Errore!")


Info: ['id', 'sdid', 'gpstime', 'sd750', 'multiEye', 'sd1500', 'sd_gpsnanotime'] 

Variabili: ['sd_theta', 'sd_phi', 'sd_energy', 'sd_l', 'sd_b', 'sd_ra', 'sd_dec', 'sd_x', 'sd_y', 'sd_z', 'sd_easting', 'sd_northing', 'sd_altitude', 'sd_s450', 'sd_beta', 'sd_gamma'] 

Incertezze: ['sd_dx', 'sd_dy', 'sd_ds450', 'sd_chi2', 'sd_ndf', 'sd_geochi2', 'sd_nbstat'] 



__OSSERVAZIONI (3)__

I veri e propri errori sulle misure sono solo quelli di posizione del nucleo dello sciame (`sd_dx`,`sd_dy`) e sull'energia di riferimento `sd_ds450`, poiché gli altri valori sono dei *filtri di qualità*.
\
Guardiamo ad esempio `sd_chi2`: esso rappresenta il $\chi{}^2$ associato alla cosiddetta _Lateral Distribution Function_ (LDF), ovvero la funzione che predice il legame tra energia dello sciame ed angolo di entrata rispetto al riferimento s450. Questo dato ci dice quanto un singolo evento rispetti la relazione del modello scelto.
\
In combinazione con il numero di gradi di libertà (rapporto `sd_chi2`/`sd_ndf`) viene fornito un valore di riferimento per decidere se quel dato è coerente con LDF oppure no. Se si sceglie una soglia bassa, si ottengono pochi dati molto precisi; al contrario, per una soglia alta si hanno molti dati ma più vari. Un dato simile è `sd_geochi2`.

Un altro filtro importante è `sd_nbstat`, che indica il numero di stazioni della griglia che registrano un segnale sopra soglia. Più questo numero è alto, più il segnale è buono. Questo filtro però è delicato dato che a sua volta dipende dall'energia e dall'angolo: sciami verticali o altissime energie alzano il numero di stazioni, ma buona parte degli eventi non sono di questo tipo.
\
Nuovamente bisogna scegliere con moderazione come filtrare per non avere pochi dati precisi o troppi ma imprecisi.

## Applicazione dei metodi ML - Regressione Lineare

Dopo aver scelto le variabili di interesse, si può passare all'implementazione di metodi ML. Il primo modello applicabile è sicuramente la __regressione lineare__.
\
Sappiamo che il metodo CIC lavora sull'energia di uno sciame e la associa con la sola componente angolare. Possiamo dunque scegliere la variabile `sd_energy` come variabile dipendente dalle altre. La regressione permetterà di trovare i parametri che minimizzano lo scarto quadratico tra i dati e la relazione lineare.

Prima di applicare la regressione lineare, andiamo a filtrare i dati secondo criteri di selezione utili al fine di non avere del rumore eccessivo. L'idea è quella di scartare tutti quei dati il cui segnale di riferimento (`sd_s450`) è poco preciso ed il rapporto `sd_chi2`/`sd_ndf` è ragionevole.

In [6]:
# Check sulle energie

low_count = mid_count = high_count = 0
pos = []
high_pos = []
low_en_dat = []

for i in range(len(mydata_var)):
    num = mydata_err[i, 3]
    den = mydata_var[i, 13]  
    if num == '':
        print("Errore `sd_ds450` mancante, ID:", mydata_info[i, 0], '\n')
        pos.append(i)
    elif den == '':
        print("Segnale `sd_s450` mancante, ID:", mydata_info[i, 0], '\n')
        pos.append(i)
        continue
    
# Cut dei segnali mancanti

for i in sorted(pos, reverse=True):
    mydata_var = np.delete(mydata_var, i, axis=0)
    mydata_err = np.delete(mydata_err, i, axis=0)
    mydata_info = np.delete(mydata_info, i, axis=0)

# Analisi range energie

for i in range(len(mydata_var)):    
    num = mydata_err[i, 3]
    den = mydata_var[i, 13] 
    sd_ratio = float(num) / float(den)
    if sd_ratio <= 2.5:
        low_count += 1
    elif sd_ratio <= 4:
        mid_count += 1
    else:
        high_count += 1
        high_pos.append(i)
        # print(high_pos[-1])
        low_en_dat.append(mydata_var[i])


tot = low_count + mid_count + high_count + len(pos) 
print("Alte energie:", low_count,
    "   Energie mid-range:", mid_count,
    "   Energie piccole:", high_count,
    "   No s450:" if tot == 54481 else '', len(pos) if tot == 54481 else '',
    "   Totale:", tot, 
    '\n'
    )

for i in sorted(high_pos, reverse=True):
    mydata_var = np.delete(mydata_var, i, axis=0)
    mydata_err = np.delete(mydata_err, i, axis=0)
    mydata_info = np.delete(mydata_info, i, axis=0)

# print(np.shape(mydata_var), '\n')

# Check su chi^2/ndf

low_count = mid_count = high_count = 0
filter_ratio = []
for i in range(len(mydata_err)):
    filter_ratio.append(float(mydata_err[i, 2])/float(mydata_err[i, -1]))
    if filter_ratio[i] <= 2.5:
        low_count += 1
    elif filter_ratio[i] >= 2.5 and filter_ratio[i] <= 4:
        mid_count += 1
    else:
        high_count += 1

tot = low_count + mid_count + high_count + len(pos) 
if tot == 54481 - len(high_pos):
    print("Dopo il taglio (`sd_ratio`<4)", tot + len(high_pos), "-->", tot - len(pos), '\n')

print("Dati precisi:", low_count,
    "   Dati mid-range:", mid_count,
    "   Rapporto grande:", high_count,
    "   Totale:", tot - len(pos),
    )



Alte energie: 53014    Energie mid-range: 1196    Energie piccole: 271    No s450: 0    Totale: 54481 

Dopo il taglio (`sd_ratio`<4) 54481 --> 54210 

Dati precisi: 54155    Dati mid-range: 11    Rapporto grande: 44    Totale: 54210


I dati eliminati sono caratterizzati dall'assenza di segnale `sd_s450`. Questo può succedere per vari motivi, ad esempio una saturazione elettronica data da un segnale ad energia troppo elevata oppure un semplice problema tecnico dei rivelatori.

Dato che la regressione lineare è un processo in cui i coefficienti hanno un significato fisico, non avrebbe senso considerare le tutte 19 variabili in `varlist` perché questo aumenterebbe la varianza del modello in maniera incoerente, ovvero dando un risultato ottimo per i dati scelti ma completamente errato per un set diverso. Per poter usare un numero maggiore di variabili si ricorre a processi di *regolarizzazione* (Ridge e LASSO).

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Regressione lineare semplice

energy = np.log10(mydata_var[:, 2].astype(float))
mydata_var = mydata_var.astype(float)
# occhio agli angoli --> cos(theta), sin(phi), cos(phi)
cos_theta = np.cos(np.radians(mydata_var[:, [np.where(myheader == 'sd_theta')[0][0]]]))
sin_phi = np.sin(np.radians(mydata_var[:, [np.where(myheader == 'sd_phi')[0][0]]]))    
cos_phi = np.cos(np.radians(mydata_var[:, [np.where(myheader == 'sd_phi')[0][0]]]))
sep_var = mydata_var.copy()
sep_var = np.delete(sep_var, [np.where(myheader == cell)[0][0] for cell in myheader if cell in ['sd_theta', 'sd_phi']], axis=1)
set_of_vars = np.concatenate((sep_var, cos_theta, sin_phi, cos_phi), axis=1)
this_varlist = varlist.copy()
this_varlist = np.delete(this_varlist, [0,1,2])
this_varlist = np.append(this_varlist, ['cos(theta)', 'sin(phi)', 'cos(phi)'])

# Definisco la parte di dataset per il training (80%)

train_size = int(len(set_of_vars) * 0.8)

# Prima provo con tutte le variabili

model1 = LinearRegression()
model1.fit(set_of_vars[:train_size, :], energy[:train_size])

print("=" * 60, '\n')
print("         Modello (1) - Tutte le variabili", '\n')
print("=" * 60, '\n')
print(f'{"Variabili fit":<20}{"Peso":<15}{"Valore":<20}')
print("-" * 50)
for i in range(len(set_of_vars[0])-1):
        print(f'{this_varlist[i]:<20}{"w_" + str(i):15}{model1.coef_[i]:<10.2e}')

print()

energy_pred = model1.predict(set_of_vars[-train_size:, :])
mse_1 = mean_squared_error(energy[-train_size:], energy_pred)
r2_1 = r2_score(energy[-train_size:], energy_pred)

print(f"\nR^2: {r2_1:.4f}")
print(f"MSE: {mse_1:.6f}")
print(f"Bias: {model1.intercept_:.4f}", '\n')

# Esclusione delle variabili non necessarie per regressione lineare. Uso solo le più grandi (10^-4)

print("=" * 60, '\n')
print("         Modello (2) - Variabili migliori", '\n')
print("=" * 60, '\n')
keep_list1 = []  
keep_idx1 = []
for i in range(1,len(set_of_vars[0])):
    if model1.coef_[i-1] >= 1.e-3:
        keep_list1.append(this_varlist[i-1])
        keep_idx1.append(i-1)

print("Variabili selezionate:", keep_list1, keep_idx1, '\n')

model2 = LinearRegression()
model2.fit(set_of_vars[:train_size, [i for i in keep_idx1]], energy[:train_size])

print(f'{"Variabili fit":<20}{"Peso":<15}{"Valore":<20}')
print("-" * 50)
for i in range(len(keep_list1)):
        print(f'{keep_list1[i]:<20}{"w_" + str(i):15}{model2.coef_[i]:<10.2e}')

print()

energy_pred = model2.predict(set_of_vars[-train_size:, [i for i in keep_idx1]])
mse_2 = mean_squared_error(energy[-train_size:], energy_pred)
r2_2 = r2_score(energy[-train_size:], energy_pred)

print(f"\nR^2: {r2_2:.4f}")
print(f"MSE: {mse_2:.6f}")
print(f"Bias: {model2.intercept_:.4f}", '\n')

# Restrizione con LASSO

print("=" * 60, '\n')
print("         Modello (3) - regolazione LASSO", '\n')
print("=" * 60, '\n')

from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# Standardizza le variabili (necessario per Lasso)
scaler = StandardScaler()
X_physical_scaled = scaler.fit_transform(set_of_vars[:train_size, keep_idx1])

# Lasso con cross-validation
lasso = LassoCV(cv=5, random_state=42, max_iter=10000, alphas=np.logspace(-4, 1, 50))
lasso.fit(X_physical_scaled, energy[:train_size])

print(f"\nAlpha ottimale: {lasso.alpha_:.6f}")

# Variabili selezionate da Lasso
selected_mask = np.abs(lasso.coef_) > 1e-6
selected_physical = [keep_list1[i] for i in range(len(keep_list1)) if selected_mask[i]]
selected_idx = [keep_idx1[i] for i in range(len(keep_idx1)) if selected_mask[i]]

print(f"\n Selezione di LASSO, {len(selected_physical)} variabili su 5:")
for v, c in zip(keep_list1, lasso.coef_):
    status = " SELEZIONATA" if abs(c) > 1e-6 else " ESCLUSA"
    print(f"  {v:<15}: {c:+.4f} (scaled) → {status}")

# Fit finale con le variabili selezionate da Lasso
if len(selected_physical) > 0:
    model3 = LinearRegression()
    model3.fit(set_of_vars[:train_size, selected_idx], energy[:train_size])
    
    energy_pred = model3.predict(set_of_vars[train_size:, selected_idx])
    r2_3 = r2_score(energy[train_size:], energy_pred)
    mse_3 = mean_squared_error(energy[train_size:], energy_pred)
    
    print(f"\nCoefficienti del modello (3) - solo variabili Lasso:")
    for v, coef in zip(selected_physical, model3.coef_):
        print(f"  {v:<15}: {coef:.4f}")
    print(f"\nR^2: {r2_3:.4f}")
    print(f"MSE: {mse_3:.6f}")
    print(f"Bias: {model3.intercept_:.4f}", '\n')
else:
    print("\n Lasso non ha selezionato nessuna variabile! Prova con alpha più piccolo.")
    model3 = None
    r2_3 = None


         Modello (1) - Tutte le variabili 


Variabili fit       Peso           Valore              
--------------------------------------------------
sd_l                w_0            1.13e-03  
sd_b                w_1            -4.30e-06 
sd_ra               w_2            1.28e-01  
sd_dec              w_3            1.56e-05  
sd_x                w_4            3.25e-05  
sd_y                w_5            -4.88e-06 
sd_z                w_6            -7.51e-05 
sd_easting          w_7            2.85e-05  
sd_northing         w_8            -3.63e-07 
sd_altitude         w_9            2.13e-07  
sd_s450             w_10           2.85e-05  
sd_beta             w_11           2.65e-04  
sd_gamma            w_12           2.15e-01  
cos(theta)          w_13           8.47e+00  
sin(phi)            w_14           -1.04e-04 
cos(phi)            w_15           -1.68e-03 


R^2: 0.5620
MSE: 0.016059
Bias: 1.2287 


         Modello (2) - Variabili migliori 


Variabili selezionate: